<a href="https://colab.research.google.com/github/StevenLevine-NOAA/NBM-Verif/blob/notebooks/probs_and_obs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#NBM "Probs and Obs" Plotter

This notebook plots shaded probabilities along with obs colored white (for hits) and black (for misses).  Ideally, we want white dots to line up with warmer colors (higher probabilities).  The notebook relies on gridded NBM data pulled from NODD (AWS archive) and observations pulled from Synoptic Data.

The idea behind this notebook is to provide quick-look verification at a selected event.  You can customize what part of the country you are looking at (by WFO) and what variable.  Note that by default, we generate a graphic for every threshold for the variable selected.

Note that this notebook works for operational NBM data only.  If you want to look at experimental NBM data, contact steven.levine@noaa.gov to gain access.

In jupyter/colab notebooks, our program is divided into a series of cells.  Each cell can be run individually.  A play button will appear at the top left corner of each cell when you mouse over it.  Click on the play button to run the relevant cell, or click "Run all" at the top to run all cells in order.

When running, you'll notice output come up below most of the cells.  This is normal.

You can look at (and edit) the code itself by clicking on the ">" to the right of each cell title.



First, we initialize the notebook, downloading and importing the relevant modules.  Part 1 ***MUST*** be run before part 2.

In [1]:
#@title Initialize Notebook Part 1
!pip install cartopy contextily pyproj pyepsg pygrib netCDF4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.8/17.8 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 36.3 MB/s eta 0:00:00


In [2]:
#@title Initialize Notebook Part 2
#!pip install cartopy contextily pyproj pyepsg pygrib netCDF4
import numpy as np
from scipy.interpolate import CubicSpline as cs, UnivariateSpline as us
from scipy.spatial import KDTree
import pandas as pd
import geopandas as gpd
from urllib.request import urlretrieve, urlopen
import requests
from datetime import datetime, timedelta
import json
from netCDF4 import Dataset
import pygrib
import pyproj
from pyproj import Proj, transform
import os, re, traceback
import math
import glob

import matplotlib
from matplotlib.colors import LinearSegmentedColormap
#from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.axes as maxes
import matplotlib.patheffects as PathEffects
from matplotlib.path import Path
from matplotlib.textpath import TextToPath
import matplotlib.gridspec as gridspec
from matplotlib.font_manager import FontProperties
from matplotlib.colors import ListedColormap
matplotlib.rcParams['font.sans-serif'] = 'Liberation Sans'
matplotlib.rcParams['font.family'] = "sans-serif"
from matplotlib.cm import get_cmap
matplotlib.use('Agg')
import seaborn as sns

from cartopy import crs as ccrs, feature as cfeature
from cartopy.io.shapereader import Reader
from cartopy.feature import ShapelyFeature
import contextily as cx
import itertools

import zipfile

from joblib import Parallel,delayed
from multiprocessing import Pool,Manager,cpu_count

from collections import defaultdict

import warnings
warnings.filterwarnings("ignore")

Now, enter information about the case you want to examine.  Use *ValidTime* and *runTime* sliders to select the relvant hour (00/06/12/18Z).  Use the calendar entries for *valid_date* and *nbm_init_date* to enter the relevant dates

When looking at a specific area, select "CWA" from region_selection, then enter the relvant CWAs in cwa_id.  CWA's should be 3 letter codes, comma separated.

When examining rain or snowfall, I find it useful to compare to stage IV or NOHRSC data instead of raw observations to get around timing issues.  Check the relevant box to do that.  Remember the URMA precip is just re-badged stage IV and URMA snowfall is re-badged NOHRSC.

By checking *export_csv*, the program will output a CSV dataFrame showing all relevant observations and interpolated probabilites.  You can examine/download them by clicking on the folder icon to the left.

In [4]:
#@title Select Options and Set Defaults

element = "qpf" #@param ["maxt","mint","temp","dewp","qpf","qpf12","qpf06","qpf48","qpf72","wind","gust","maxwind","maxgust","snow","snow06","snow48","snow72","vis","fire"]
valid_date = "2026-08-18" #@param {type:"date"}
ValidTime = 0 #@param {type:"slider", min:0, max:18, step:6}
use_stageiv = False #@param {type:"boolean"}
use_nohrsc = False #@param {type:"boolean"}
nbm_init_date = "2026-08-17" #@param {type:"date"}
nbm_init_hour = 0 #@param {type:"slider", min:0, max:18, step:6}
region_selection = "CWA" #@param ["WR", "SR", "CR", "ER", "AR", "CONUS", "CWA"]
cwa_id = "HFO" #@param {type:"string"}
network_selection = "ALL" #@param ["NWS", "RAWS", "NWS+RAWS", "NWS+RAWS+HADS", "ALL", "CUSTOM", "LIST"]
cwa_outline = True #@param {type:"boolean"}
county_outline = False #@param {type:"boolean"}
export_csv = True #@param {type:"boolean"}
#@markdown Light or dark theme plots?
plot_style = "dark" #@param ["light", "dark"]

if region_selection == "CONUS":
  region_list = ["WR", "CR", "SR", "ER"]
elif region_selection == "CWA":
  region_list = [cwa_id]
#elif region_selection == "AR":
  #region_list=["AJK","ARH","AFC"]
else:
  region_list = [region_selection]

def cwa_list(input_region):
  region_dict ={"WR":"BYZ,BOI,LKN,EKA,FGZ,GGW,TFX,VEF,LOX,MFR,MTR,MSO,PDT,PSR,PIH,PQR,REV,STO,SLC,SGX,HNX,SEW,OTX,TWC",
              "CR":"ABR,BIS,CYS,LOT,DVN,BOU,DMX,DTX,DDC,DLH,FGF,GLD,GJT,GRR,GRB,GID,IND,JKL,EAX,ARX,ILX,LMK,MQT,MKX,MPX,LBF,APX,IWX,OAX,PAH,PUB,UNR,RIW,FSD,SGF,LSX,TOP,ICT",
              "ER":"ALY,LWX,BGM,BOX,BUF,BTV,CAR,CTP,RLX,CHS,ILN,CLE,CAE,GSP,MHX,OKX,PHI,PBZ,GYX,RAH,RNK,AKQ,ILM",
              "SR":"ABQ,AMA,FFC,EWX,BMX,BRO,CRP,EPZ,FWD,HGX,HUN,JAN,JAX,KEY,MRX,LCH,LZK,LUB,MLB,MEG,MFL,MOB,MAF,OHX,LIX,OUN,SJT,SHV,TAE,TBW,TSA",
              "AR":"AJK,ARH,AFC"}
  if (input_region in ["WR", "CR", "SR", "ER","AR"]):
    cwas_list = region_dict[input_region]
  else:
    cwas_list = input_region
  return cwas_list

nbm_init = datetime.strptime(nbm_init_date,'%Y-%m-%d') + timedelta(hours=int(nbm_init_hour))

if element == "maxt":
    nbm_core_valid_hour="00"
    nbm_qmd_valid_hour="06"
    valid_date_start = datetime.strptime(valid_date,'%Y-%m-%d')
    valid_date_end = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(days=1)
    obs_start_hour = "1200"
    obs_end_hour = "0600"
    ob_stat = "maximum"
    valid_end_datetime = valid_date_end + timedelta(hours=(int(obs_end_hour)/100))
    nbm_core_valid_end_datetime = valid_date_end + timedelta(hours=int(nbm_core_valid_hour))
    nbm_qmd_valid_end_datetime = valid_date_end + timedelta(hours=int(nbm_qmd_valid_hour))
    core_init = nbm_init + timedelta(hours = 7)
    nbm_core_fhdelta = nbm_core_valid_end_datetime - core_init
    nbm_qmd_fhdelta = nbm_qmd_valid_end_datetime - nbm_init

elif element == "mint":
    nbm_core_valid_hour="12"
    nbm_qmd_valid_hour="18"
    valid_date_start = datetime.strptime(valid_date,'%Y-%m-%d')
    valid_date_end = datetime.strptime(valid_date,'%Y-%m-%d')
    obs_start_hour = "0000"
    obs_end_hour = "1800"
    ob_stat = "minimum"
    valid_end_datetime = valid_date_end + timedelta(hours=(int(obs_end_hour)/100))
    nbm_core_valid_end_datetime = valid_date_end + timedelta(hours=int(nbm_core_valid_hour))
    nbm_qmd_valid_end_datetime = valid_date_end + timedelta(hours=int(nbm_qmd_valid_hour))
    core_init = nbm_init + timedelta(hours = 7)
    nbm_core_fhdelta = nbm_core_valid_end_datetime - core_init
    nbm_qmd_fhdelta = nbm_qmd_valid_end_datetime - nbm_init

elif element == "qpf":
    nbm_core_valid_hour = (str(ValidTime)).zfill(2)
    nbm_valid_hour = (str(ValidTime)).zfill(2)
    nbm_qmd_valid_hour=(str(ValidTime)).zfill(2)
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(hours=int(ValidTime))
    valid_date_start = valid_date - timedelta(hours=24)
    valid_date_end = valid_date
    obs_start_hour = (str(ValidTime)).zfill(2)+"00"
    obs_end_hour = (str(ValidTime)).zfill(2)+"00"
    ob_stat = "total"
    valid_end_datetime = valid_date_end
    core_init = nbm_init
    nbm_core_valid_end_datetime = valid_date_end
    nbm_qmd_valid_end_datetime = valid_date_end
    nbm_core_fhdelta = nbm_core_valid_end_datetime - nbm_init
    nbm_qmd_fhdelta = nbm_qmd_valid_end_datetime - nbm_init

elif element == "qpf06":
    nbm_core_valid_hour = (str(ValidTime)).zfill(2)
    nbm_valid_hour = (str(ValidTime)).zfill(2)
    nbm_qmd_valid_hour=(str(ValidTime)).zfill(2)
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(hours=int(ValidTime))
    valid_date_start = valid_date - timedelta(hours=6)
    valid_date_end = valid_date
    obs_start_hour = (str(ValidTime)).zfill(2)+"00"
    obs_end_hour = (str(ValidTime)).zfill(2)+"00"
    ob_stat = "total"
    valid_end_datetime = valid_date_end
    core_init = nbm_init
    nbm_core_valid_end_datetime = valid_date_end
    nbm_qmd_valid_end_datetime = valid_date_end
    nbm_core_fhdelta = nbm_core_valid_end_datetime - nbm_init
    nbm_qmd_fhdelta = nbm_qmd_valid_end_datetime - nbm_init

elif element == "qpf12":
    nbm_core_valid_hour = (str(ValidTime)).zfill(2)
    nbm_valid_hour = (str(ValidTime)).zfill(2)
    nbm_qmd_valid_hour=(str(ValidTime)).zfill(2)
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(hours=int(ValidTime))
    valid_date_start = valid_date - timedelta(hours=12)
    valid_date_end = valid_date
    obs_start_hour = (str(ValidTime)).zfill(2)+"00"
    obs_end_hour = (str(ValidTime)).zfill(2)+"00"
    ob_stat = "total"
    valid_end_datetime = valid_date_end
    core_init = nbm_init
    nbm_core_valid_end_datetime = valid_date_end
    nbm_qmd_valid_end_datetime = valid_date_end
    nbm_core_fhdelta = nbm_core_valid_end_datetime - nbm_init
    nbm_qmd_fhdelta = nbm_qmd_valid_end_datetime - nbm_init

elif element == "qpf48":
    nbm_core_valid_hour = (str(ValidTime)).zfill(2)
    nbm_valid_hour = (str(ValidTime)).zfill(2)
    nbm_qmd_valid_hour=(str(ValidTime)).zfill(2)
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(hours=int(ValidTime))
    valid_date_start = valid_date - timedelta(hours=48)
    valid_date_end = valid_date
    obs_start_hour = (str(ValidTime)).zfill(2)+"00"
    obs_end_hour = (str(ValidTime)).zfill(2)+"00"
    ob_stat = "total"
    valid_end_datetime = valid_date_end
    core_init = nbm_init
    nbm_core_valid_end_datetime = valid_date_end
    nbm_qmd_valid_end_datetime = valid_date_end
    nbm_core_fhdelta = nbm_core_valid_end_datetime - nbm_init
    nbm_qmd_fhdelta = nbm_qmd_valid_end_datetime - nbm_init

elif element == "qpf72":
    nbm_core_valid_hour = (str(ValidTime)).zfill(2)
    nbm_valid_hour = (str(ValidTime)).zfill(2)
    nbm_qmd_valid_hour=(str(ValidTime)).zfill(2)
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(hours=int(ValidTime))
    valid_date_start = valid_date - timedelta(hours=72)
    valid_date_end = valid_date
    obs_start_hour = (str(ValidTime)).zfill(2)+"00"
    obs_end_hour = (str(ValidTime)).zfill(2)+"00"
    ob_stat = "total"
    valid_end_datetime = valid_date_end
    core_init = nbm_init
    nbm_core_valid_end_datetime = valid_date_end
    nbm_qmd_valid_end_datetime = valid_date_end
    nbm_core_fhdelta = nbm_core_valid_end_datetime - nbm_init
    nbm_qmd_fhdelta = nbm_qmd_valid_end_datetime - nbm_init

elif element == "vis":
    nbm_core_valid_hour=(str(ValidTime)).zfill(2)
    nbm_qmd_valid_hour=(str(ValidTime)).zfill(2)
    nbm_valid_hour=(str(ValidTime)).zfill(2)
    obs_start_hour=(str(ValidTime)).zfill(2)+"00"
    obs_end_hour=(str(ValidTime)).zfill(2)+"00"
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(hours=int(ValidTime))
    valid_date_start = valid_date
    valid_date_end = valid_date
    valid_end_datetime=valid_date_end
    core_init=nbm_init
    nbm_core_valid_end_datetime=valid_date_end
    nbm_core_fhdelta=nbm_core_valid_end_datetime-core_init
    nbm_qmd_valid_end_datetime=valid_date_end

elif element in ["maxwind","maxgust"]:
    #nbm_core_valid_hour="06"
    #nbm_valid_hour="06"
    nbm_qmd_valid_hour="06"
    obs_start_hour="0600"
    obs_end_hour="0600"
    ob_stat="maximum"
    valid_date_start = datetime.strptime(valid_date,'%Y-%m-%d')
    valid_date_end = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(days=1)
    valid_end_datetime=valid_date_end + timedelta(hours=(int(obs_end_hour)/100))
    core_init = nbm_init
    nbm_core_valid_end_datetime = valid_date_end
    nbm_qmd_valid_end_datetime = valid_date_end + timedelta(hours=int(nbm_qmd_valid_hour))
    nbm_core_fhdelta = valid_end_datetime - nbm_init
    nbm_qmd_fhdelta = nbm_qmd_valid_end_datetime - nbm_init
    #valid_date=date.strptime(valid_date,'') + timedelta(hours=)

elif element in ["wind","gust","temp","dewp","fire"]:
    nbm_core_valid_hour=(str(ValidTime)).zfill(2)
    nbm_qmd_valid_hour=(str(ValidTime)).zfill(2)
    obs_start_hour=(str(ValidTime)).zfill(2)+"00"
    obs_end_hour=(str(ValidTime)).zfill(2)+"00"
    ob_stat="nearest"
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(hours=int(ValidTime))
    valid_date_start = valid_date
    valid_date_end = valid_date
    valid_end_datetime=valid_date_end
    core_init = nbm_init
    nbm_core_valid_end_datetime = valid_date_end
    nbm_qmd_valid_end_datetime = valid_date_end
    nbm_core_fhdelta = nbm_core_valid_end_datetime - core_init
    nbm_qmd_fhdelta = nbm_qmd_valid_end_datetime - nbm_init

elif element == "snow":# or element == "ice":
    nbm_core_valid_hour=(str(ValidTime)).zfill(2)
    nbm_qmd_valid_hour=(str(ValidTime)).zfill(2)
    obs_start_hour=(str(ValidTime)).zfill(2)+"00"
    obs_end_hour = (str(ValidTime)).zfill(2)+"00"
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d')+ timedelta(hours=int(ValidTime))
    valid_date_start = valid_date - timedelta(hours=24)
    valid_date_end = valid_date
    valid_end_datetime = valid_date_end #+ timedelta(hours=(int(obs_end_hour)/100))
    ob_stat = "total"
    core_init = nbm_init + timedelta(hours = 1)
    nbm_qmd_valid_end_datetime = valid_date_end #+ timedelta(hours=int(nbm_qmd_valid_hour))
    nbm_core_valid_end_datetime = nbm_qmd_valid_end_datetime
    nbm_core_fhdelta = nbm_core_valid_end_datetime - core_init

elif element == "snow06":# or element == "ice":
    nbm_core_valid_hour=(str(ValidTime)).zfill(2)
    nbm_qmd_valid_hour=(str(ValidTime)).zfill(2)
    obs_start_hour=(str(ValidTime)).zfill(2)+"00"
    obs_end_hour = (str(ValidTime)).zfill(2)+"00"
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d')+ timedelta(hours=int(ValidTime))
    valid_date_start = valid_date - timedelta(hours=6)
    valid_date_end = valid_date
    valid_end_datetime = valid_date_end #+ timedelta(hours=(int(obs_end_hour)/100))
    ob_stat = "total"
    core_init = nbm_init + timedelta(hours = 1)
    nbm_qmd_valid_end_datetime = valid_date_end #+ timedelta(hours=int(nbm_qmd_valid_hour))
    nbm_core_valid_end_datetime = nbm_qmd_valid_end_datetime
    nbm_core_fhdelta = nbm_core_valid_end_datetime - core_init

elif element == "snow48":# or element == "ice":
    nbm_core_valid_hour=(str(ValidTime)).zfill(2)
    nbm_qmd_valid_hour=(str(ValidTime)).zfill(2)
    obs_start_hour=(str(ValidTime)).zfill(2)+"00"
    obs_end_hour = (str(ValidTime)).zfill(2)+"00"
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d')+ timedelta(hours=int(ValidTime))
    valid_date_start = valid_date - timedelta(hours=48)
    valid_date_end = valid_date
    valid_end_datetime = valid_date_end #+ timedelta(hours=(int(obs_end_hour)/100))
    ob_stat = "total"
    core_init = nbm_init + timedelta(hours = 1)
    nbm_qmd_valid_end_datetime = valid_date_end #+ timedelta(hours=int(nbm_qmd_valid_hour))
    nbm_core_valid_end_datetime = nbm_qmd_valid_end_datetime
    nbm_core_fhdelta = nbm_core_valid_end_datetime - core_init

elif element == "snow72":# or element == "ice":
    nbm_core_valid_hour=(str(ValidTime)).zfill(2)
    nbm_qmd_valid_hour=(str(ValidTime)).zfill(2)
    obs_start_hour=(str(ValidTime)).zfill(2)+"00"
    obs_end_hour = (str(ValidTime)).zfill(2)+"00"
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d')+ timedelta(hours=int(ValidTime))
    valid_date_start = valid_date - timedelta(hours=72)
    valid_date_end = valid_date
    valid_end_datetime = valid_date_end #+ timedelta(hours=(int(obs_end_hour)/100))
    ob_stat = "total"
    core_init = nbm_init + timedelta(hours = 1)
    nbm_qmd_valid_end_datetime = valid_date_end #+ timedelta(hours=int(nbm_qmd_valid_hour))
    nbm_core_valid_end_datetime = nbm_qmd_valid_end_datetime
    nbm_core_fhdelta = nbm_core_valid_end_datetime - core_init

else:
    raise ValueError(f"Element not recognized: {element}")

current_datetime=datetime.now()


#if element == "snow" or element == "qpf":
#  nbm_core_forecasthour_start = nbm_core_forecasthour - 24
#elif element in "vis":
#  nbm_core_forecasthour_start = nbm_core_forecasthour
#else:
#  nbm_core_forecasthour_start = nbm_core_forecasthour - 12

nbm_qmd_fhdelta = nbm_qmd_valid_end_datetime - nbm_init
nbm_qmd_forecasthour = nbm_qmd_fhdelta.total_seconds() / 3600.
nbm_core_forecasthour = nbm_core_fhdelta.total_seconds() / 3600.
if element == "qpf" or element == "maxwind" or element == "maxgust" or element == "snow" or element == "ice24":
  nbm_qmd_forecasthour_start = nbm_qmd_forecasthour - 24
  nbm_core_forecastHour_start = nbm_core_forecasthour - 24
elif element == "qpf12":
  nbm_qmd_forecasthour_start = nbm_qmd_forecasthour - 12
  nbm_core_forecasthour_start = nbm_core_forecasthour - 12
elif element == "qpf06":
  nbm_qmd_forecasthour_start = nbm_qmd_forecasthour - 6
  nbm_core_forecasthour_start = nbm_core_forecasthour - 6
elif element in ["qpf48","snow48"]:
  nbm_qmd_forecasthour_start = nbm_qmd_forecasthour - 48
  nbm_core_forecasthour_start = nbm_core_forecasthour - 48
elif element in ["qpf72","snow72"]:
  nbm_qmd_forecasthour_start = nbm_qmd_forecasthour - 72
  nbm_core_forecasthour_start = nbm_core_forecasthour - 72
elif element in ["vis","temp","dewp","fire","wind","gust"]:
  nbm_qmd_forecasthour_start = nbm_qmd_forecasthour
  nbm_core_forecasthour_start = nbm_core_forecasthour
else:
  print(f"WARNING: Forecast length unknown for element {element}.  Defaulting to 18")
  nbm_qmd_forecasthour_start = nbm_qmd_forecasthour - 18
  nbm_core_forecasthour_start = nbm_core_forecasthour - 18

# Setup a dictionary for translating a form selection into a something we can pass to mesowest API
network_dict = {"NWS+RAWS+HADS":"&network=1,2,106","NWS+RAWS":"&network=1,2", "NWS":"&network=1", "RAWS": "&network=2", "ALL":""}
network_string = network_dict[network_selection]

if use_stageiv and element=="qpf":
  points_str = f'Stage IV @ {network_selection}'
else:
  points_str = network_selection

if plot_style=="light":
  background_color = '#f7f7f7'
  text_color = '#121212'
  map_land_color = '#FAFAF8'
  map_water_color = '#D4DBDD'
  map_border_color = 'grey'
elif plot_style=="dark":
  background_color = '#272727'
  text_color = 'white'
  map_land_color = '#414143'
  map_water_color = '#272727'
  #map_border_color = '#3B3B3D'
  map_border_color = 'white'

########################################################################################################################
# Reusable functions section                                                                                           #
########################################################################################################################

def project3(lon, lat, prj):
  lon = float(lon)
  lat = float(lat)

  outproj = prj
  inproj = Proj(init='epsg:4326')
  nbm_coords = transform(inproj, outproj, lon, lat)
  coordX = nbm_coords[0]
  coordY = nbm_coords[1]
  #print(f'Lat: {lat}, Y: {coordY} | Lon: {lon}, X: {coordX}')
  return(coordX, coordY)


def ll_to_index(datalons, datalats, loclon, loclat):
  abslat = np.abs(datalats-loclat)
  abslon = np.abs(datalons-loclon)
  c = np.maximum(abslon, abslat)
  latlon_idx_flat = np.argmin(c)
  latlon_idx = np.unravel_index(latlon_idx_flat, datalons.shape)
  return(latlon_idx)


def project_hrap(lon, lat, s4x, s4y):
  lon = float(lon)
  lat = float(lat)

  globe = ccrs.Globe(semimajor_axis=6371200)
  hrap_ccrs = proj = ccrs.Stereographic(central_latitude=90.0,
                          central_longitude=255.0,
                          true_scale_latitude=60.0, globe=globe)
  latlon_ccrs = ccrs.PlateCarree()
  hrap_coords = hrap_ccrs.transform_point(lon,lat,src_crs=latlon_ccrs)
  hrap_idx = ll_to_index(s4x, s4y, hrap_coords[0], hrap_coords[1])

  return hrap_idx

def nohrsc_ll2ij(lon,lat,gridlons,gridlats):
  #for a lat/lon grid
  lon = float(lon)
  lat = float(lat)
  lonidx=(np.abs(lon-gridlons)).argmin()
  latidx=(np.abs(lat-gridlats)).argmin()
  return(latidx,lonidx)

def get_stageiv():
  siv_url = "https://water.noaa.gov/resources/downloads/precip/stageIV/"+valid_date_end.strftime('%Y')+"/"+valid_date_end.strftime('%m')+"/"+valid_date_end.strftime('%d')+"/nws_precip_1day_"+valid_date_end.strftime('%Y%m%d')+"_conus.nc"
  siv_file = "nws_precip_1day_"+valid_date_end.strftime('%Y%m%d')+"_conus.nc"
  urlretrieve(siv_url,siv_file)
  nc = Dataset(siv_file,'r')
  stageIV = nc.variables['observation']
  s4x = nc.variables['x']
  s4y = nc.variables['y']
  return stageIV, s4x, s4y

def get_urma_precip(valtime,region,numhrs):
  year=valtime.strftime('%Y')
  month=valtime.strftime('%m')
  day=valtime.strftime('%d')
  hour=valtime.strftime('%H')
  tdHours=numhrs-6
  print(f"Looking back {tdHours} hours from initial time for URMA precip")
  startTime=valtime-timedelta(hours=tdHours)
  dateRange=pd.date_range(start=startTime,end=valtime,freq='6H')
  precipVals={}
  if not os.path.isdir('urma'):
    os.mkdir('urma')
  for date in dateRange:
    #print(f"Getting precip for {date}")
    yearStr=date.strftime('%Y')
    monthStr=date.strftime('%m')
    dayStr=date.strftime('%d')
    hourStr=date.strftime('%H')
    print(f"Pulling URMA precip data for: {monthStr}/{dayStr}/{yearStr} {hourStr}Z")

    if "AFC" in region or "AFG" in region or "AJK" in region:
      fileName=f'pcpurma_ak.{yearStr}{monthStr}{dayStr}{hourStr}.06h.grb2'
      url=f'https://noaa-urma-pds.s3.amazonaws.com/akurma.{yearStr}{monthStr}{dayStr}/{fileName}'
    elif region == "SJU":
      fileName=f'pcpurma_pr.{yearStr}{monthStr}{dayStr}{hourStr}.06h.grb2'
      url=f'https://noaa-urma-pds.s3.amazonaws.com/prurma.{yearStr}{monthStr}{dayStr}/{fileName}'
    else:
      print(f"Defaulting region to CONUS.")
      fileName=f'urma2p5.{yearStr}{monthStr}{dayStr}{hourStr}.pcp_06h.wexp.grb2'
      url=f'https://noaa-urma-pds.s3.amazonaws.com/urma2p5.{yearStr}{monthStr}{dayStr}/{fileName}'

    joinedPath=os.path.join('urma',fileName)
    try:
      if os.path.exists(joinedPath):
        print(f'Already found {fileName}')
      else:
        print(f'Downloading {url}')
        urlretrieve(url,joinedPath)
        print(f'Successfully downloaded {fileName}')
      precipGrib=pygrib.open(joinedPath)
      precip=precipGrib.select(name='Total Precipitation')[0]
      precipVals[date]=mm_to_in(precip.values)
      lats,lons=precip.latlons()
      #precipGrib.close()
    except Exception as e:
      print(f'Error downloading {fileName}: {e}')
      if region == "co":
        precipVals[date]=np.zeros((1597,2345))
      elif region == "pr":
        precipVals[date]=np.zeros((1649,1105))
      elif region == "ak":
        precipVals[date]=np.zeros((339,225))
      else:
        print(f"Unknown region {region}.  Nothing to read")
        break
  totalPrecip=np.zeros_like(lats)
  for date in dateRange:
    totalPrecip+=precipVals[date]
  return totalPrecip,lats,lons

def get_nohrsc():
  nohrsc_url = "https://www.nohrsc.noaa.gov/snowfall_v2/data/"+valid_date_end.strftime('%Y%m')+"/sfav2_CONUS_24h_"+valid_date_end.strftime('%Y%m%d%H')+".nc"
  data = urlopen(nohrsc_url).read()

  nc = Dataset('data',memory=data)
  snow=np.asarray(nc.variables['Data']) #make lon by lat array (original lat by lon)
  snowlat = np.asarray(nc.variables['lat'])
  snowlon = np.asarray(nc.variables['lon'])
  return snow,snowlon,snowlat

def K_to_F(kelvin):
  fahrenheit = 1.8*(kelvin-273)+32.
  return fahrenheit

def C_to_F(celsius):
  fahrenheit = 1.8*celsius+32.
  return fahrenheit

def mps_to_kts(mps):
  kts = mps * 1.94384
  return kts

def mps_to_mph(mps):
  mph = mps * 2.23694
  return mph

def mph_to_ms(mph):
  mps = mph * 0.4470
  return mps

def mm_to_in(millimeters):
  inches = millimeters * 0.0393701
  return inches

def meters_to_in(meters):
  inches = meters*39.3701
  return inches

def find_roots(x,y):
  s = np.abs(np.diff(np.sign(y))).astype(bool)
  return x[:-1][s] + np.diff(x)[s]/(np.abs(y[1:][s]/y[:-1][s])+1)

class FileSizeError(Exception):
  """Raised when a file exists but its size is invalid for processing."""
  pass


Now that we've set our options, lets download what we need from NODD and Synoptic!

First, we download the relevant observations from Synoptic Data.

In [5]:
#@title Download Obs
from google.colab import userdata
synoptic_token=userdata.get('synoptic_token')

statistics_api = "https://api.synopticlabs.org/v2/stations/legacystats?"
precipitation_api = "https://api.synopticdata.com/v2/stations/precipitation?"
metadata_api = "https://api.synopticdata.com/v2/stations/metadata?"
nearest_api = "https://api.synopticdata.com/v2/stations/nearesttime?"

print('Getting obs...')
obs ={}
mandate=valid_date_end

for region in region_list:
  if (valid_end_datetime <= current_datetime):
    print("   > Grabbing obs for: ", region)
    #print("List of CWAs: ", cwa_list(region) )
    json_name = "obs/Obs_"+element+"_"+valid_date_start.strftime('%Y%m%d')+obs_start_hour+"_"+valid_date_end.strftime('%Y%m%d')+obs_end_hour+"_"+region+".json"
    if os.path.exists("obs"):
      pass
    else:
      os.mkdir("obs")
    if element in ["mint","maxt"]:
      api_token = "&token="+synoptic_token
      station_query = "&cwa="+cwa_list(region)
      vars_query = "&vars=air_temp"
      start_query = "&start="+valid_date_start.strftime('%Y%m%d')+obs_start_hour
      end_query = "&end="+valid_date_end.strftime('%Y%m%d')+obs_end_hour
      stat_type = "&type="+ob_stat
      network_query = network_string
      api_extras = "&units=temp%7Cf&within=1440&status=active"
      obs_url = statistics_api + api_token + station_query + vars_query + start_query + end_query + stat_type + network_query + api_extras
    elif element == "maxwind":
      api_token = "&token="+synoptic_token
      station_query = "&cwa="+cwa_list(region)
      vars_query = "&vars=wind_speed"
      start_query = "&start="+valid_date_start.strftime('%Y%m%d')+obs_start_hour
      end_query = "&end="+valid_date_end.strftime('%Y%m%d')+obs_end_hour
      stat_type = "&type="+ob_stat
      network_query = network_string
      obs_url = statistics_api + api_token + station_query + vars_query + start_query + end_query + stat_type + network_query
    elif element == "vis":
      api_token = "&token="+synoptic_token
      station_query = "&cwa="+cwa_list(region)
      vars_query="&vars=visibility"
      start_query = "&attime="+valid_date_start.strftime('%Y%m%d%H')+"00&within=60"
      #end_query=""
      network_query=network_string
      #units_extras = ""
      api_extras = "&status=active"
      obs_url = nearest_api + api_token + station_query + vars_query + start_query + network_query + api_extras
    elif "qpf" in element:
      if use_stageiv:
        api_token = "&token="+synoptic_token
        station_query = "&cwa="+cwa_list(region)
        api_extras = "&fields=status,latitude,longitude,name,elevation"
        network_query = network_string
        obs_url = metadata_api + api_token + station_query + network_query + api_extras
        if element == "qpf12":
          stageIV,sivlats,sivlons = get_urma_precip(valid_end_datetime,region,12)
        elif element == "qpf06":
          stageIV,sivlats,sivlons = get_urma_precip(valid_end_datetime,region,6)
        elif element == "qpf48":
          stageIV,sivlats,sivlons = get_urma_precip(valid_end_datetime,region,48)
        elif element == "qpf72":
          stageIV,sivlats,sivlons = get_urma_precip(valid_end_datetime,region,72)
        elif element == "qpf":
          stageIV,sivlats,sivlons = get_urma_precip(valid_end_datetime,region,24)
        else:
          print(f"Warning: Unknown QPF time from element {element} with use_stageiv == True.  Defaulting to qpf24...")
          stageIV,sivlats,sivlons = get_urma_precip(valid_end_datetime,region,12)
        #stageIVlons,stageIVlats = np.meshgrid(sivlons,sivlats)
        #stageIV, s4xs, s4ys = get_stageiv()
        #s4xs, s4ys = np.meshgrid(s4xs, s4ys)
      else:
        api_token = "&token="+synoptic_token
        station_query = "&cwa="+cwa_list(region)
        api_extras = "&fields=status,latitude,longitude,name,elevation&obtimezone=utc"
        network_query = network_string
        vars_query = "&pmode=totals"
        units_query = "&units=precip|in"
        start_query = "&start="+valid_date_start.strftime('%Y%m%d')+obs_start_hour
        end_query = "&end="+valid_date_end.strftime('%Y%m%d')+obs_end_hour
        obs_url = precipitation_api + api_token + station_query + network_query + vars_query + start_query + end_query + units_query + api_extras
    elif "snow" in element:
      if use_nohrsc:
        api_token = "&token="+synoptic_token
        station_query = "&cwa="+cwa_list(region)
        api_extras = "&fields=status,latitude,longitude,name,elevation"
        network_query = network_string
        obs_url = metadata_api + api_token + station_query + network_query + api_extras
        snow,snowlon,snowlat = get_nohrsc()
        snowlons,snowlats = np.meshgrid(snowlon,snowlat)
      else:
        api_token = "&token="+synoptic_token
        station_query = "&cwa="+cwa_list(region)
        api_extras = "&fields=status,latitude,longitude,name,elevation&obtimezone=utc"
        network_query = network_string
        vars_query = "&pmode=totals"
        units_query = "&units=precip|in"
        start_query = "&start="+valid_date_start.strftime('%Y%m%d')+obs_start_hour
        end_query = "&end="+valid_date_end.strftime('%Y%m%d')+obs_end_hour
        obs_url = precipitation_api + api_token + station_query + network_query + vars_query + start_query + end_query + units_query + api_extras
    else: #element in ["wind","gust","temp","dewp","fire"]
      if element == "wind":
        vars_query="&vars=wind_speed"
      elif element == "gust":
        vars_query="&vars=wind_gust"
      elif element == "temp":
        vars_query="&vars=air_temp"
      elif element == "dewp":
        vars_query="&vars=dew_point_temperature"
      elif element == "vis":
        vars_query="&vars=visibility"
      elif element == "fire":
        vars_query="&vars=wind_speed,relative_humidity,wind_gust"
      else:
        raise ValueError(f"Element not recognized: {element}")
      at_query="&attime="+valid_date_start.strftime('%Y%m%d%H')+"00&within=60"
      api_token = "&token="+synoptic_token
      station_query = "&cwa="+cwa_list(region)
      network_query = network_string
      obs_url = nearest_api + api_token + station_query + vars_query + at_query + network_query

    if os.path.exists(json_name):
      print ("Skipping download since JSON file already exists")
      pass
    else:
      urlretrieve(obs_url, json_name)

    if os.path.exists(json_name):
      if os.path.getsize(json_name) > 500:
        with open(json_name) as json_file:
            obs_json = json.load(json_file)
            print ("Loaded Obs JSON file line 343: " + json_name)
            obs_lats = []
            obs_lons = []
            if element == "fire":
              obs_rh_values=[]
              obs_wspd_values=[]
              obs_wgust_values=[]
            else:
              obs_value = []
            obs_elev = []
            obs_stid = []
            obs_name = []
            obs_time = []
            obs_network = []
            numStns=len(obs_json["STATION"])
            print(f"Reading in and interpolating {numStns} obs")
            for stn in obs_json["STATION"]:
                if stn["STID"] is None:
                  stid = "N0N3"
                else:
                  stid = stn["STID"]
                name = stn["NAME"]
                if stn["ELEVATION"] and stn["ELEVATION"] is not None:
                  elev = stn["ELEVATION"]
                else:
                  elev = -999
                if "MNET_ID" in stn.keys():
                  ntwk=stn["MNET_ID"]
                else:
                  ntwk=np.nan
                lat = stn["LATITUDE"]
                lon = stn["LONGITUDE"]
                if float(lon) > -50:
                  continue #bug fix to deal with errant synoptic labs obs in the file
                if element in ["mint","maxt"]:
                  if 'air_temp_set_1' in stn['STATISTICS'] and stn['STATISTICS']['air_temp_set_1']:
                    if ob_stat in stn['STATISTICS']['air_temp_set_1']: # and float(stn["LATITUDE"]) != 0. and float(stn["LONGITUDE"]) != 0.:
                      stat = stn['STATISTICS']['air_temp_set_1'][ob_stat]
                      obs_stid.append(str(stid))
                      obs_name.append(str(name))
                      obs_elev.append(float(elev))
                      obs_lats.append(float(lat))
                      obs_lons.append(float(lon))
                      obs_value.append(float(stat))
                      obs_time.append(date)
                      try:
                        obs_network.append(ntwk)
                      except:
                        obs_network.append(np.nan)
                elif element == "temp":
                  if 'air_temp_value_1' in stn['OBSERVATIONS'] and stn['OBSERVATIONS']['air_temp_value_1']:
                    value=stn["OBSERVATIONS"]["air_temp_value_1"]["value"]
                    if 'date_time' in stn['OBSERVATIONS']['air_temp_value_1']:
                      obdate=stn['OBSERVATIONS']['air_temp_value_1']['date_time']
                      date=datetime.strptime(obdate,'%Y-%m-%dT%H:%M:%SZ')
                    else:
                      date=mandate
                    obs_stid.append(str(stid))
                    obs_name.append(str(name))
                    obs_elev.append(float(elev))
                    obs_lats.append(float(lat))
                    obs_lons.append(float(lon))
                    obs_value.append(float(C_to_F(float(value))))
                    obs_time.append(date)
                    try:
                      obs_network.append(ntwk)
                    except:
                      obs_network.append(np.nan)
                elif element == "dewp":
                  if 'dew_point_temperature_value_1' in stn['OBSERVATIONS'] and stn['OBSERVATIONS']['dew_point_temperature_value_1']:
                    value=stn["OBSERVATIONS"]["dew_point_temperature_value_1"]["value"]
                    date=stn["OBSERVATIONS"]["dew_point_temperature_value_1"]["date_time"]
                  elif 'dew_point_temperature_value_1d' in stn['OBSERVATIONS'] and stn['OBSERVATIONS']['dew_point_temperature_value_1d']:
                    value=stn["OBSERVATIONS"]["dew_point_temperature_value_1d"]["value"]
                    date=stn["OBSERVATIONS"]["dew_point_temperature_value_1d"]["date_time"]
                  else:
                    continue
                  obs_stid.append(str(stid))
                  obs_name.append(str(name))
                  obs_elev.append(float(elev))
                  obs_lats.append(float(lat))
                  obs_lons.append(float(lon))
                  obs_value.append(float(C_to_F(float(value))))
                  obs_time.append(datetime.strptime(date,"%Y-%m-%dT%H:%M:%SZ"))
                  try:
                    obs_network.append(ntwk)
                  except:
                    obs_network.append(np.nan)
                elif element == "maxwind":
                  if 'wind_speed_set_1' in stn['STATISTICS'] and stn['STATISTICS']['wind_speed_set_1']:
                    if ob_stat in stn['STATISTICS']['wind_speed_set_1']: # and float(stn["LATITUDE"]) != 0.:
                      stat = stn['STATISTICS']['wind_speed_set_1'][ob_stat]
                      obs_stid.append(str(stid))
                      if 'date_time' in stn['STATISTICS']['wind_speed_set_1']:
                        obdate=stn['STATISTICS']['wind_speed_set_1']['date_time']
                        date=datetime.strptime(obdate,'%Y-%m-%dT%H:%M:%SZ')
                      else:
                        date=mandate
                      obs_name.append(str(name))
                      obs_elev.append(float(elev))
                      obs_lats.append(float(lat))
                      obs_lons.append(float(lon))
                      obs_value.append(mps_to_kts(float(stat)))
                      obs_time.append(date)
                      try:
                        obs_network.append(ntwk)
                      except:
                        obs_network.append(np.nan)
                elif element == "maxgust":
                  if 'wind_gust_set_1' in stn['STATISTICS'] and stn['STATISTICS']['wind_gust_set_1']:
                    if ob_stat in stn['STATISTICS']['wind_gust_set_1'] and float(stn["LATITUDE"]) > 0. and float(stn["LONGITUDE"] < 0.):
                      stat = stn['STATISTICS']['wind_gust_set_1'][ob_stat]
                      if 'date_time' in stn['STATISTICS']['wind_gust_set_1']:
                        obdate=stn['STATISTICS']['wind_gust_set_1']['date_time']
                        date=datetime.strptime(obdate,'%Y-%m-%dT%H:%M:%SZ')
                      else:
                        date=mandate
                      obs_stid.append(str(stid))
                      obs_name.append(str(name))
                      obs_elev.append(float(elev))
                      obs_lats.append(float(lat))
                      obs_lons.append(float(lon))
                      obs_value.append(mps_to_kts(float(stat)))
                      obs_time.append(date)
                      try:
                        obs_network.append(ntwk)
                      except:
                        obs_network.append(np.nan)
                elif element in ["wspd","wind"]:
                  if stn["STATUS"] == "ACTIVE" and float(stn["LATITUDE"]) > 0. and float(stn["LONGITUDE"]) < 0.:
                    if 'wind_speed_value_1' in stn['OBSERVATIONS'] and stn['OBSERVATIONS']['wind_speed_value_1']:
                      obval=stn['OBSERVATIONS']['wind_speed_value_1']['value']
                      if 'date_time' in stn['OBSERVATIONS']['wind_speed_value_1']:
                        obdate=stn['OBSERVATIONS']['wind_speed_value_1']['date_time']
                        date=datetime.strptime(obdate,'%Y-%m-%dT%H:%M:%SZ')
                      else:
                        date=mandate
                      obs_stid.append(str(stid))
                      obs_name.append(str(name))
                      obs_elev.append(float(elev))
                      obs_lats.append(float(lat))
                      obs_lons.append(float(lon))
                      obs_value.append(mps_to_kts(float(obval)))
                      obs_time.append(date)
                      try:
                        obs_network.append(ntwk)
                      except:
                        obs_network.append(np.nan)
                elif element in ["wgust","gust"]:
                  if stn["STATUS"] == "ACTIVE" and float(stn["LATITUDE"]) > 0. and float(stn["LONGITUDE"]) < 0.:
                    if 'wind_gust_value_1' in stn['OBSERVATIONS'] and stn['OBSERVATIONS']['wind_gust_value_1']:
                      obval=stn['OBSERVATIONS']['wind_gust_value_1']['value']
                      if 'date_time' in stn['OBSERVATIONS']['wind_gust_value_1']:
                        obdate=stn['OBSERVATIONS']['wind_gust_value_1']['date_time']
                        date=datetime.strptime(obdate,'%Y-%m-%dT%H:%M:%SZ')
                      else:
                        date=mandate
                      obs_stid.append(str(stid))
                      obs_name.append(str(name))
                      obs_elev.append(float(elev))
                      obs_lats.append(float(lat))
                      obs_lons.append(float(lon))
                      obs_value.append(mps_to_kts(float(obval)))
                      obs_time.append(date)
                      try:
                        obs_network.append(ntwk)
                      except:
                        obs_network.append(np.nan)
                elif element == "vis":
                  if 'visibility_value_1' in stn['OBSERVATIONS'] and stn['OBSERVATIONS']['visibility_value_1']:
                    if (stn["STATUS"] == "ACTIVE"):
                      obs_stid.append(str(stid))
                      obs_name.append(str(name))
                      obs_elev.append(float(elev))
                      obs_lats.append(float(lat))
                      obs_lons.append(float(lon))
                      obs_value.append(float(stn['OBSERVATIONS']['visibility_value_1']['value']))
                      obs_time.append(date)
                      try:
                        obs_network.append(ntwk)
                      except:
                        obs_network.append(np.nan)
                elif "qpf" in element:
                  if (stn["STATUS"] == "ACTIVE"): # and float(stn["LATITUDE"]) < 50.924 and float(stn["LATITUDE"]) > 23.377 and float(stn["LONGITUDE"]) > -125.650 and float(stn["LONGITUDE"]) < -66.008:
                    obs_stid.append(str(stid))
                    obs_name.append(str(name))
                    obs_elev.append(float(elev))
                    obs_lats.append(float(lat))
                    obs_lons.append(float(lon))
                    obs_network.append(ntwk)
                    obs_time.append(mandate)
                    #if use_stageiv:
                    #  ivcoords=ll_to_index(sivlons,sivlats,float(lon),float(lat))
                    #  ptotal=stageIV[ivcoords]
                    #  if ptotal >= 0.01:
                    #    obs_value.append(ptotal)
                    #  elif ptotal < 0.0:
                    #    print(f"WARNING: Negative URMA precip value for site {stid}: {ptotal}")
                    #    obs_value.append(np.nan)
                    #  elif np.isnan(ptotal):
                    #    print(f"WARNING: NaN URMA precip value  for site {stid}")
                    #    obs_value.append(np.nan)
                    #  else:
                    #    obs_value.append(0.0)
                    if use_stageiv is False:
                      if "precipitation" in stn["OBSERVATIONS"]:
                        if "total" in stn["OBSERVATIONS"]["precipitation"][0]:
                          ptotal = stn["OBSERVATIONS"]["precipitation"][0]["total"]
                          print(f"Adding observed {element} value: {ptotal}.")
                          if ptotal >= 0.005:
                            obs_value.append(ptotal)
                          else:
                            obs_value.append(0.0)
                        else:
                          obs_value.append(np.nan)
                      else:
                        obs_value.append(np.nan)
                elif "snow" in element:
                  if stn["STATUS"] == "ACTIVE": # and float(stn["LATITUDE"]) < 50.924)and float(stn["LATITUDE"]) > 23.377 and float(stn["LONGITUDE"]) > -125.650 and float(stn["LONGITUDE"]) < -66.008:
                    obs_stid.append(str(stid))
                    obs_name.append(str(name))
                    obs_elev.append(float(elev))
                    obs_lats.append(float(lat))
                    obs_lons.append(float(lon))
                    obs_network.append(ntwk)
                    obs_time.append(date)
                    if use_nohrsc:
                      coords = nohrsc_ll2ij(lon,lat,snowlon,snowlat)
                      nohrsc_value = meters_to_in(float(snow[coords]))
                      if nohrsc_value >= 0.005:
                        obs_value.append(nohrsc_value)
                      elif nohrsc_value < 0.0:
                        obs_value.append(np.nan)
                      else:
                        obs_value.append(0.0)
                    else:
                      #raise Exception("Still not able to process individual snow obs!")
                      if "precipitation" in stn["OBSERVATIONS"]:
                        if "total" in stn["OBSERVATIONS"]["precipitation"][0] and stn["OBSERVATIONS"]["precipitation"][0] is not None:
                          ptotal = stn["OBSERVATIONS"]["precipitation"][0]["total"]
                          if (float(ptotal) >= 0.01):
                            obs_value.append(mm_to_in(float(ptotal)))
                          else:
                            obs_value.append(0.00)
                        else:
                          obs_value.append(np.nan)
                elif element == "fire":
                  if stn["STATUS"] == "ACTIVE" and float(stn["LATITUDE"]) > 0. and float(stn["LONGITUDE"]) < 0.:
                    #fireFound=False
                    obs_stid.append(str(stid))
                    obs_name.append(str(name))
                    obs_elev.append(float(elev))
                    obs_lats.append(float(lat))
                    obs_lons.append(float(lon))
                    obs_time.append(np.nan)
                    try:
                      obs_network.append(ntwk)
                    except:
                      obs_network.append(np.nan)
                    if 'wind_speed_value_1' in stn['OBSERVATIONS'] and stn['OBSERVATIONS']['wind_speed_value_1']:
                      wspd=stn['OBSERVATIONS']['wind_speed_value_1']['value']
                      obs_wspd_values.append(mps_to_mph(float(wspd)))
                      #fireFound=True
                    else:
                      obs_wspd_values.append(np.nan)
                    if 'relative_humidity_value_1' in stn['OBSERVATIONS'] and stn['OBSERVATIONS']['relative_humidity_value_1']:
                      rh=stn['OBSERVATIONS']['relative_humidity_value_1']['value']
                      obs_rh_values.append(float(rh))
                    else:
                      obs_rh_values.append(np.nan)
                    if 'wind_gust_value_1' in stn['OBSERVATIONS'] and stn['OBSERVATIONS']['wind_gust_value_1']:
                      wgust=stn['OBSERVATIONS']['wind_gust_value_1']['value']
                      obs_wgust_values.append(mps_to_mph(float(wgust)))
                    else:
                      obs_wgust_values.append(np.nan)
                else:
                  raise ValueError (f"Unsupported element found in readobs: {element}")
            csv_name = "obs_"+element+"_"+region+"_"+valid_date_end.strftime('%Y%m%d')+".csv"
            print(f"Number of stations: {len(obs_stid)}")
            obs[region] = pd.DataFrame()
            obs[region]["stid"] = obs_stid
            obs[region]["name"] = obs_name
            obs[region]["elevation"] = obs_elev
            obs[region]["lat"] = obs_lats
            obs[region]["lon"] = obs_lons
            obs[region]["network"] = obs_network
            obs[region]["datetime"] = obs_time
            if element == "fire":
              obs[region]["ob_rh"] = obs_rh_values
              obs[region]["ob_wspd_mph"] = obs_wspd_values
              obs[region]["ob_wgust_mph"] = obs_wgust_values
              obs[region]["ob_rh"]=pd.to_numeric(obs[region]["ob_rh"],errors='coerce')
              obs[region]["ob_wspd_mph"]=pd.to_numeric(obs[region]["ob_wspd_mph"],errors='coerce')
              obs[region]["ob_wgust_mph"]=pd.to_numeric(obs[region]["ob_wgust_mph"],errors='coerce')
            elif "ob_"+element in obs[region].columns:
              obs[region]["ob_"+element] = obs_value
              obs[region]["ob_"+element] = pd.to_numeric(obs[region]["ob_"+element],errors='coerce')
            else:
              print(f"WARNING: Not loading obs for element ob_{element}.  This is OK if this is a QPF ob")
            obs[region]["elevation"] = pd.to_numeric(obs[region]["elevation"],errors='coerce')
            obs[region]["lat"] = pd.to_numeric(obs[region]["lat"],errors='coerce')
            obs[region]["lon"] = pd.to_numeric(obs[region]["lon"],errors='coerce')
            obs[region]["network"] = pd.to_numeric(obs[region]["network"],errors='coerce')
            obs[region]["datetime"] = pd.to_datetime(obs[region]["datetime"])
            print(f"First length of obs dataFrame: {len(obs[region])}")
            #obs[region].dropna(inplace=True)
            print(f"Length of obs dataFrame: {len(obs[region])}")
            #obs[region].to_csv(csv_name)
      else:
        raise FileSizeError(f"File is too small to read: {json_name}")
    else:
      raise FileNotFoundError(f"JSON ob file {json_name} not found!")
  else:
    print(f'    > Valid Time in the future. Grabbing obs points only for: {region}')
    json_name = "obs/ObsPoints_"+region+"_wcoss.json"
    if os.path.exists(json_name):
      pass
    else:
      if os.path.exists("obs"):
        pass
      else:
        os.mkdir("obs")
      obs_url = "https://api.synopticdata.com/v2/stations/metadata?&token="+synoptic_token+"&cwa="+cwa_list(region)+"&fields=status,latitude,longitude,name,elevation"+network_string
      urlretrieve(obs_url, json_name)
    if os.path.exists(json_name):
      with open(json_name) as json_file:
          obs_json = json.load(json_file)
          print("Loaded Obs JSON file line 197!")
          obs_lats = []
          obs_lons = []
          obs_elev = []
          obs_stid = []
          obs_name = []
          for stn in obs_json["STATION"]:
            # print(stn.encode('utf-8'))
            if stn["STID"] is None:
              stid = "N0N3"
            else:
              stid = stn["STID"]
            #print(f'Processing {region} station {stid}')
            name = stn["NAME"]
            if stn["ELEVATION"] and stn["ELEVATION"] is not None:
              elev = stn["ELEVATION"]
            else:
              elev = -999
            lat = stn["LATITUDE"]
            lon = stn["LONGITUDE"]
            if stn["STATUS"] == "ACTIVE": # and float(stn["LATITUDE"]) != 0. and float(stn["LONGITUDE"]) != 0.:
              obs_stid.append(str(stid))
              obs_name.append(str(name))
              obs_elev.append(float(elev))
              obs_lats.append(float(lat))
              obs_lons.append(float(lon))
          obs[region] = pd.DataFrame()
          obs[region]["stid"] = obs_stid
          obs[region]["name"] = obs_name
          obs[region]["elevation"] = obs_elev
          obs[region]["lat"] = obs_lats
          obs[region]["lon"] = obs_lons
          obs[region]["ob_"+element] = -999
          #obs[region].dropna(inplace=True)
          #obs[region].to_csv(csv_name)

#end up with one master frame of all obs
masterobs=pd.concat(obs.values(),ignore_index=True)
obFileName=csv_name
masterobs.to_csv(obFileName)

Getting obs...
   > Grabbing obs for:  HFO
Loaded Obs JSON file line 343: obs/Obs_qpf_202608170000_202608180000_HFO.json
Reading in and interpolating 470 obs
Adding observed qpf value: 0.0.
Adding observed qpf value: 0.0.
Adding observed qpf value: 0.208.
Adding observed qpf value: 1.98.
Adding observed qpf value: 0.0.
Adding observed qpf value: 0.314.
Adding observed qpf value: 0.002.
Adding observed qpf value: 0.0.
Adding observed qpf value: 0.001.
Adding observed qpf value: 0.94.
Adding observed qpf value: 5.02.
Adding observed qpf value: 0.4.
Adding observed qpf value: 0.43.
Adding observed qpf value: 0.85.
Adding observed qpf value: 0.07.
Adding observed qpf value: 0.01.
Adding observed qpf value: 0.04.
Adding observed qpf value: 1.04.
Adding observed qpf value: 0.0.
Adding observed qpf value: 0.04.
Adding observed qpf value: 0.35.
Adding observed qpf value: 0.51.
Adding observed qpf value: 0.02.
Adding observed qpf value: 0.64.
Adding observed qpf value: 2.06.
Adding observed qpf

Now, we cut those obs to the relevant CWAs where applicable.

In [ ]:
#@title Cut Obs to relevant CWA

#try:
if not os.path.exists("shp/w_05mr24.shp"):
    cwa_url = "https://www.weather.gov/source/gis/Shapefiles/WSOM/w_18mr25.zip" # Updated URL
    if not os.path.exists("shp"):
      os.mkdir("shp")
    urlretrieve(cwa_url, "shp/nws_cwa_outlines.zip")
with zipfile.ZipFile("shp/nws_cwa_outlines.zip", 'r') as zip_ref:
      zip_ref.extractall("shp")
      cwa_shapes=gpd.read_file("shp/w_18mr25.shp")
      cwaList=cwa_id.split(',')
      cwa_subset=cwa_shapes[cwa_shapes['WFO'].isin(cwaList)]
      min_lon, min_lat, max_lon, max_lat = cwa_subset.total_bounds
      min_lon=min_lon-2
      min_lat=min_lat-2
      max_lon=max_lon+2
      max_lat=max_lat+2
      #cwa_subset.to_file("shp/cwa_subset.shp")
      filteredObs=masterobs[(masterobs['lat'] > min_lat) & (masterobs['lat'] < max_lat) & (masterobs['lon'] > min_lon) & (masterobs['lon'] < max_lon)]
#except:
  #print("WARNING: Cannot download CWA files to filter obs!")
  #filteredObs=obs.copy()
for region in region_list:
    #filteredName="obs_"+element+"_"+region+"_filtered.csv"
    filteredName=f"obs_{element}_{region}_{valid_date_end.strftime('%Y%m%d')}_filtered.csv"
    filteredObs.to_csv(filteredName)
    print(f"For region {region}: Original obs {len(masterobs)}, filtered obs {len(filteredObs)}")

Now we download the relevant NBM data from NODD/AWS.

Note the download_subset subroutine, which ensure we only download the data we need, rather than the whole (very large) grib2 file.

In [ ]:
#@title Download Gridded Data

def getthresh(element):
	if element == "maxgust":
		#kts:m/s
		idthresh = {
			22:11,
			34:17,
			41:21,
			48:24,
			56:28,
			64:32
		}
	elif element == "maxwind":
		#kts:m/s
		idthresh = {
			11:5,
			17:8,
			22:11,
			34:17,
			48:24,
			64:32
		}
	elif element == "wind":
		idthresh={
			7:3.601,
			11:5.6588,
			17:8.7456,
			22:11.317699999999999,
			30:15.4333,
			34:17.4911,
			48:24.6933,
			64:32.924400000000006
		}
	elif element == "gust":
		idthresh={
			17:8.7456,
			22:11.317699999999999,
			30:15.4333,
			34:17.4911,
			41:21.092200000000002,
			48:24.6933,
			56:28.808900000000005,
			64:32.924400000000006
		}
	elif element == "maxt":
		#F:K
		idthresh = {
			0:255.372,
			28:270,
			32:273.15,
			80:299.8169,
			90:305.372,
			100:310.928,
			110:316.483,
			120:322.03900000000004
		}
	elif element == "mint":
		#F:K
		idthresh = {
			-40:233,
			-20:244,
			0:255,
			10:260,
			28:270.928,
			32:273.15
			#80:299
		}
	elif element == "dewp":
	#F:K
		idthresh= {
			0:255.372,
			10:260.928,
			20:266.483,
			32:273.15,
			45:280.372,
			50:283.15,
			55:285.928,
			60:288.706,
			65:291.483,
			70:294.26090000000005,
			75:297.03900000000004,
			80:299.81690000000003
		}
	elif element == "temp":
		idthresh = {
			0:255.372,
			28:270.928,
			32:273.15,
			80:299.81690000000003,
			90:305.372,
			100:310.928,
			110:316.483,
			120:322.03900000000004
		}
	elif element == "qpf": #24-H QPF
		#in to mm
		idthresh = {
			0.10:0.254,
			0.25:6.35,
			0.50:12.7,
			1.00:25.4,
			2.00:50.8,
			3.00:76.2,
			4.00:101.6,
			5.00:127.0,
			6.00:152.4,
			8.00:203.2
		}
	elif element == "qpf06":
		#in to mm
		idthresh = {
			0.01:0.254,
			0.10:2.54,
			0.25:6.35,
			0.50:12.7,
			0.75:19.05,
			1.00:25.4,
			1.50:38.1,
			2.00:50.8,
			2.50:63.8,
			3.00:76.2
		}
	elif element == "qpf12":
		#in to mm
		idthresh = {
				 0.01:0.254,
				 0.10:2.54,
				 0.25:6.35,
				 0.50:12.7,
				 1.00:25.4,
				 2.00:50.8,
				 3.00:76.2,
				 4.00:101.6,
				 5.00:127.0,
				 8.00:203.2
		}
	elif element == "qpf48":
		idthresh = {
			0.1:2.54,
			0.5:5.08,
			1.0:25.4,
			2.0:50.8,
			3.0:76.2,
			5.0:127.0,
			8.0:203.2,
			10.0:254.0,
			12.0:304.8,
			15.0:381.0
		}
	elif element == "qpf72":
		idthresh = {
			0.25:6.35,
			1.0:25.4,
			2.0:50.8,
			3.0:76.2,
			5.0:127.0,
			8.0:203.2,
			10.0:254.0,
			15.0:381.0,
			20.0:508.0,
			25.0:635.0
		}
	elif element == "snow": #snow 24
		idthresh = {
			0.1:0.0025399999999999997,
			#0.3:0.00508,
			#0.5:0.0127,
			#0.7:0.0178,
			1.0:0.0254,
			#1.5:0.0381,
			2.0:0.0508,
			#2.5:0.0638,
			3.0:0.0762,
			4.0:0.1016,
			6.0:0.1524,
			8.0:0.2032,
			10.0:0.254,
			12.0:0.3048,
			18.0:0.4572,
			24.0:0.6096,
			#30.0:0.7620,
			#36.0:0.9144,
			#48.0:1.2192
		}
	elif element == "snow06":
		idthresh = {
			0.1:0.1,
			0.3:0.3,
			0.5:0.5,
			0.7:0.7,
			1.0:1.0,
			1.5:1.5,
			2.0:2.0,
			2.5:2.5,
			3.0:3.0,
			4.0:4.0,
			5.0:5.0,
			6.0:6.0,
			8.0:8.0,
			10.0:10.0,
			12.0:12.0,
			18.0:18.0,
			24.0:24.0,
			30.0:30.0,
			36.0:36.0,
			48.0:48.0
		}
	elif element == "snow48":
		idthresh = {
			0.1:0.1,
			0.3:0.3,
			0.5:0.5,
			0.7:0.7,
			1.0:1.0,
			2.0:2.0,
			2.5:2.5,
			3.0:3.0,
			4.0:4.0,
			5.0:5.0,
			6.0:6.0,
			8.0:8.0,
			10.0:10.0,
			12.0:12.0,
			18.0:18.0,
			24.0:24.0,
			36.0:36.0,
			48.0:48.0
		}
	elif element == "snow72":
		idthresh = {
			0.1:0.1,
			0.3:0.3,
			0.5:0.5,
			0.7:0.7,
			1.0:1.0,
			2.0:2.0,
			2.5:2.5,
			3.0:3.0,
			4.0:4.0,
			5.0:5.0,
			6.0:6.0,
			8.0:8.0,
			10.0:10.0,
			12.0:12.0,
			18.0:18.0,
			24.0:24.0,
			36.0:36.0,
			48.0:48.0
		}
	elif element in ["ice06","ice"]:
		idthresh = {
			0.01:0.01,
			0.1:0.1,
			0.25:0.25,
			0.50:0.50,
			1.00:1.00
		}
	elif element in ["vis"]:
		idthresh = {
				1:1609.34,
				2:3218.69,
				3:4828.03,
				5:8046.73
		}
	else:
		raise Exception ("No thresholds in place yet for element " + element + ".  Look at convert.py to add thresholds as needed.")
	return idthresh

def fireThresh(element):
	#In grib2 files, we will simply go by message number, as joint prob values are not specified in grib2 message
	#We use defaultDicts here, since we are using either wind or gust, depending on the message number to prevent key errors.
	if element == "fire":
		windThresh=defaultdict(lambda: np.nan)
		gustThresh=defaultdict(lambda: np.nan)
		rhThresh=defaultdict(lambda: np.nan)
		rhDict= {
				 0:35.0,
				 1:30.0,
				 2:25.0,
				 3:25.0,
				 4:25.0,
				 5:20.0,
				 6:20.0,
				 7:20.0,
				 8:15.0,
				 9:15.0,
				 10:10.0,
				 11:35.0,
				 12:25.0,
				 13:25.0,
				 14:15.0,
				 15:15.0
		}
		windDict= {
				 0:10.0,
				 1:15.0,
				 2:10.0,
				 3:15.0,
				 4:20.0,
				 5:15.0,
				 6:20.0,
				 7:30.0,
				 8:15.0,
				 9:20.0,
				 10:30.0
		}
		gustDict= {
				 11:25.0,
				 12:30.0,
				 13:55.0,
				 14:25.0,
				 15:35.0
		}
		rhThresh.update(rhDict)
		windThresh.update(windDict)
		gustThresh.update(gustDict)
		return windThresh,gustThresh,rhThresh
	else:
		raise ValueError (f"Called fire weather threshold subroutine for element: {element}")

def download_subset(remote_url, remote_file, local_filename):
  print("   > Downloading a subset of NBM gribs")
  local_file = "nbm/"+local_filename
  if "qmd" in remote_file:
    if element == "maxt":
      if (int(nbm_qmd_forecasthour_start) % 24 == 0) and (int(nbm_qmd_forecasthour) % 24 ==0):
        search_string = f':TMP:2 m above ground:{str(int(int(nbm_qmd_forecasthour_start)/24))}-{str(int(int(nbm_qmd_forecasthour)/24))} day max fcst:'
      else:
        search_string = f':TMP:2 m above ground:{str(int(nbm_qmd_forecasthour_start))}-{str(int(nbm_qmd_forecasthour))} hour max fcst:'
    elif element == "mint":
      if (int(nbm_qmd_forecasthour_start) % 24 == 0) and (int(nbm_qmd_forecasthour) % 24 ==0):
        search_string = f':TMP:2 m above ground:{str(int(int(nbm_qmd_forecasthour_start)/24))}-{str(int(int(nbm_qmd_forecasthour)/24))} day min fcst:'
      else:
        search_string = f':TMP:2 m above ground:{str(int(nbm_qmd_forecasthour_start))}-{str(int(nbm_qmd_forecasthour))} hour min fcst:'
    elif "qpf" in element:
      if (int(nbm_qmd_forecasthour_start) % 24 == 0) and (int(nbm_qmd_forecasthour) % 24 ==0):
        search_string = f':APCP:surface:{str(int(int(nbm_qmd_forecasthour_start)/24))}-{str(int(int(nbm_qmd_forecasthour)/24))} day acc fcst:'
      else:
        search_string = f':APCP:surface:{str(int(nbm_qmd_forecasthour_start))}-{str(int(nbm_qmd_forecasthour))} hour acc fcst:'
    elif element == "maxwind":
      if (int(nbm_qmd_forecasthour_start) % 24 == 0) and (int(nbm_qmd_forecasthour) % 24 == 0):
        search_string = f':WIND:10 m above ground:{str(int(nbm_qmd_forecasthour_start/24))}-{str(int(nbm_qmd_forecasthour/24))} day max fcst:'
      else:
        search_string = f':WIND:10 m above ground:{str(int(nbm_qmd_forecasthour_start))}-{str(int(nbm_qmd_forecasthour))} hour max fcst:'
    elif element == "maxgust":
      if (int(nbm_qmd_forecasthour_start) % 24 == 0) and (int(nbm_qmd_forecasthour) % 24 == 0):
        search_string = f':GUST:10 m above ground:{str(int(nbm_qmd_forecasthour_start/24))}-{str(int(nbm_qmd_forecasthour/24))} day max fcst:'
      else:
        search_string = f':GUST:10 m above ground:{str(int(nbm_qmd_forecasthour_start))}-{str(int(nbm_qmd_forecasthour))} hour max fcst:'
    elif element == "temp":
      search_string = f':TMP:2 m above ground:{str(int(nbm_qmd_forecasthour))} hour fcst:'
    elif element == "dewp":
      search_string = f':DPT:2 m above ground:{str(int(nbm_qmd_forecasthour))} hour fcst:'
    elif element == "gust":
      search_string = f':GUST:10 m above ground:{str(int(nbm_qmd_forecasthour))} hour fcst:'
    elif element == "wind":
      search_string = f':WIND:10 m above ground:{str(int(nbm_qmd_forecasthour))} hour fcst:'
    elif element == "fire":
      search_string = f':JFWPRB:surface:{str(int(nbm_qmd_forecasthour))} hour fcst:'
  elif "core" in remote_file:
    if element == "maxt":
      search_string = f':TMAX:2 m above ground:{str(int(nbm_core_forecasthour_start))}-{str(int(nbm_core_forecasthour))} hour max fcst:'
    elif element == "mint":
      search_string = f':TMIN:2 m above ground:{str(int(nbm_core_forecasthour_start))}-{str(int(nbm_core_forecasthour))} hour min fcst:'
    elif "snow in element":
      search_string = f':ASNOW:surface:{str(int(nbm_core_forecasthour_start))}-{str(int(nbm_core_forecasthour))} hour acc'
    elif element == "vis":
      search_string = f':VIS:surface:{str(int(nbm_core_forecasthour_start))} hour fcst:'


  #print("Search string = ",search_string)
  idx = remote_url+".idx"
  #print("IDX file = " + idx)
  r = requests.get(idx)
  if not r.ok:
    print('     ❌ SORRY! Status Code:', r.status_code, r.reason)
    print(f'      ❌ It does not look like the index file exists: {idx}')

  lines = r.text.split('\n')
  expr = re.compile(search_string)
  expr
  byte_ranges = {}
  for n, line in enumerate(lines, start=1):
    # n is the line number (starting from 1) so that when we call for
    # `lines[n]` it will give us the next line. (Clear as mud??)
    # Use the compiled regular expression to search the line
    #print(">> Searching throgh this line: " + line)
    if expr.search(line):
      # aka, if the line contains the string we are looking for...
      # Get the beginning byte in the line we found
      parts = line.split(':')
      rangestart = int(parts[1])
      # Get the beginning byte in the next line...
      if n+1 < len(lines):
        # ...if there is a next line
        parts = lines[n].split(':')
        rangeend = int(parts[1])
      else:
        # ...if there isn't a next line, then go to the end of the file.
        rangeend = ''

        # Store the byte-range string in our dictionary,
        # and keep the line information too so we can refer back to it.
      byte_ranges[f'{rangestart}-{rangeend}'] = line
      #print(line)
    #else:
      #print(">>>  Could not find search string!")
  #print(">>  Number of items in byteRange:" + str(len(byte_ranges)))
  for i, (byteRange, line) in enumerate(byte_ranges.items()):

    if i == 0:
      # If we are working on the first item, overwrite the existing file.
      curl = f'curl -s --range {byteRange} {remote_url} > {local_file}'
      #print(">>  Adding curl command: " + curl)
    else:
      # If we are working on not the first item, append the existing file.
      curl = f'curl -s --range {byteRange} {remote_url} >> {local_file}'
      #print("Adding curl command: " + curl)
    #print('>>  Parsing line: ' + line)
    try:
      num, byte, date, var, level, forecast, _ = line.split(':')
    except:
      pass
      #print(">>>  Can't get num/byte/etc from this line, so skipping...")

    #print(f'  Downloading GRIB line [{num:>3}]: variable={var}, level={level}, forecast={forecast}')
    #print(f'  Downloading GRIB line: variable={var}, level={level}, forecast={forecast}')
    #print("Running the curl command...")
    os.system(curl)

  if os.path.exists(local_file):
    print(f'      ✅ Success! Searched for [{search_string}] and got [{len(byte_ranges)}] GRIB fields and saved as {local_file}')
    return local_file
  else:
    print(print(f'      ❌ Unsuccessful! Searched for [{search_string}] and did not find anything!'))

########################################################################################################################
# This section downloads and processes the NBM.                                                                        #
########################################################################################################################
if "AR" in region_list or "AJK" in region_list or "ARH" in region_list or "AFC" in region_list:
  rg="ak"
elif "HFO" in region_list:
  rg="hi"
elif "SJU" in region_list:
  rg="pr"
else:
  rg="co"

print('Getting and processing NBM...')
nbm_init_filen = nbm_init.strftime('%Y%m%d') + "_" + nbm_init.strftime('%H')
nbm_init_filen_core = core_init.strftime('%Y%m%d') + "_" + core_init.strftime('%H')
nbm_url_base = "https://noaa-nbm-grib2-pds.s3.amazonaws.com/blend."+nbm_init.strftime('%Y%m%d') \
            +"/"+nbm_init.strftime('%H')+"/"
nbm_url_base_core = "https://noaa-nbm-grib2-pds.s3.amazonaws.com/blend."+core_init.strftime('%Y%m%d') \
            +"/"+core_init.strftime('%H')+"/"
temp_vars = ["maxt","mint"]

prob_dict = {"maxt":"maxt18p", "mint":"mint18p", "qpf":"qpf24p", "snow":"snow24p"}
if element in ["snow","vis"]:
  prob_file=f"blend.t{int(core_init.strftime('%H')):02}z.core.f{int(nbm_core_forecasthour):03}.{rg}.grib2"
  prob_url=nbm_url_base_core+"core/"+prob_file
else:
  prob_file = f'blend.t{int(nbm_init_hour):02}z.qmd.f{int(nbm_qmd_forecasthour):03}.{rg}.grib2'
  prob_url = nbm_url_base+"qmd/"+prob_file
prob_file_subset = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{rg}.{element}_subset.grib2'
print("prob_url=",prob_url)
print("prob_file=",prob_file)
if os.path.exists("nbm"):
  pass
else:
  os.mkdir("nbm")
#Raise exception for missinge percentile data
if rg in ["pr","hi"] and element in ["maxt","mint","maxwind","snow"]:
  raise Exception ("FATAL ERROR: Percentiles for " + element + "do not exist for PR or HI")
if os.path.exists("nbm/"+prob_file_subset):
  print("   > NBM probabilistic already exists")
else:
  #urlretrieve(perc_url, "nbm/"+perc_file)
  print("   > Getting NBM probabilistic")
  download_subset(prob_url, prob_file, prob_file_subset)



Now we interpolate the gridded probabilities to our observations, creating a dataFrame and exporting that to a CSV file if desired.

In [ ]:
#@title Parse Downloaded Probabilities to Obs (without map), complete DataFrame

nbmprob = pygrib.open("nbm/"+prob_file_subset)
print('   > Extracting NBM Probabilistic')

if element == "fire":
  windThresh,gustThresh,rhThresh=fireThresh(element)
  #nbmprob.seek(2)
  #tmpinv=nbmprob.read(2)[0]
  #nbmlats,nbmlons=tmpinv.latlons()
else:
  idthresh=getthresh(element)
probfield={}
point_lats=filteredObs["lat"]
point_lons=filteredObs["lon"]

#initial read of one random field for lat/lon info and identify index/grid values
print ("Interpolating ob locations...")
nbm_fidx=[]
nbmprob.seek(2)
tmpinv=nbmprob.read(2)[0]
nbmlats,nbmlons=tmpinv.latlons()
obsPoints=np.stack([point_lats,point_lons],axis=-1)
gridPoints=np.stack([nbmlats.flatten(),nbmlons.flatten()],axis=-1)
tree=KDTree(gridPoints)
dist,flatIndex=tree.query(obsPoints,p=np.inf)
unraveledIndicies=np.unravel_index(flatIndex,nbmlats.shape)
nbm_fidx=list(zip(unraveledIndicies[0],unraveledIndicies[1]))
#for i in range(0,len(point_lats)):
#  coords = ll_to_index(nbmlons,nbmlats,point_lons[i],point_lats[i])
#  nbm_fidx.append(coords)
filteredObs["NBM_fidx"] = nbm_fidx
print("Resetting prob file...")
nbmprob.seek(0)

#now read probabilities and add to dataframe
if element == "fire":
  for thresh in rhThresh.keys():
    rhVal=rhThresh[thresh]
    print("Extracting for RH threshold: " + str(thresh) + "/" + str(rhVal))
    windVal=windThresh[thresh]
    print("Extracting for wind threshold: " + str(thresh) + "/" + str(windVal))
    gustVal=gustThresh[thresh]
    print("Extracting for gust threshold: " + str(thresh) + "/" + str(gustVal))
    try:
      probinv=nbmprob[thresh+1]
    except:
      print("Can't find data for this threshold")
      continue
    if math.isnan(windVal):
      prob_name=f"Prob RH < {rhVal} and Wind Gust > {gustVal}"
    elif math.isnan(gustVal):
      prob_name=f"Prob RH < {rhVal} and Wind Speed > {windVal}"
    else:
      raise ValueError (f"Both gust and wind have valid values for message number {thresh}")
    probdata = probinv.values
    nbm_coords = filteredObs["NBM_fidx"].values
    prob_values = []
    for i in range(0, len(nbm_coords)):
      prob_value = probdata[nbm_coords[i]]
      prob_values.append(prob_value)
    filteredObs[prob_name] = prob_values
    probfield[thresh] = probdata
  nbmprob.close()
else:
  #Interpolate URMA/Stage IV precip values here to save processing time, assuming URMA and NBM_fidx coordinates are identical
  if 'qpf' in element and use_stageiv is True:
    if "ob_"+element in filteredObs.columns:
      print(f"WARNING: Interpolating Stage IV/URMA to ob points despite obs already being in dataFrame.  Current obs will be overwritten!")
    else:
      print(f"Interpolating URMA to ob points...")
    obs_values=[]
    for coord in nbm_fidx:
      stnid4urma=filteredObs.loc[filteredObs['NBM_fidx'] == coord, 'stid']
      #It is possible that more than one station will have same coordinates.  If so, we will pick one ID to be used for debugging/logging purposes only.
      numIds=len(stnid4urma)
      if len(stnid4urma) > 1:
        urmaID=stnid4urma.iloc[0]
      else:
        urmaID=stnid4urma.iloc[0]
      ptotal=stageIV[coord]
      if ptotal >= 0.01:
        obs_values.append(ptotal)
      elif ptotal < 0.0:
        print(f"WARNING: Negative URMA precip value: {ptotal} at station {urmaID} URMA coords {coord}")
        obs_values.append(np.nan)
      elif np.isnan(ptotal):
        print(f"WARNING: NAN URMA precip value: {ptotal} at station {urmaID} URMA coords {coord}")
        obs_values.append(np.nan)
      else:
        #print(f"WARNING: Unknown URMA precip value: {ptotal} at station {urmaID} URMA coords {coord}")
        obs_values.append(0.0)
    filteredObs["ob_"+element] = obs_values
    print("URMA/Stage IV interpolation complete, obs loaded into dataFrame")
  #now intepolate probabilities to obs (all variables)
  for thresh in idthresh.keys():
    tval=idthresh[thresh]
    print(f"Extracting for threshold: {thresh}/{tval}")
    if element == "maxt":
      limit=tval
      if limit < 280:
        try:
          probinv = nbmprob.select(name="Time-maximum 2 metre temperature",probabilityTypeName="Probability of event below lower limit",lowerLimit=limit)[0]
        except:
          print("Can't find data for this threshold")
          continue
        prob_name="Prob " + element + " < " + str(thresh)
      else:
        try:
          probinv = nbmprob.select(name="Time-maximum 2 metre temperature",probabilityTypeName="Probability of event above upper limit",upperLimit=limit)[0]
        except:
          print("Can't find data for this threshold")
          continue
        prob_name="Prob " + element + " > " + str(thresh)
      probdata = probinv.values
    elif element == "mint":
      limit=tval
      if limit < 280:
        try:
          probinv = nbmprob.select(name="Time-minimum 2 metre temperature",probabilityTypeName="Probability of event below lower limit",lowerLimit=limit)[0]
        except:
          print("Can't find data for this threshold")
          continue
        prob_name="Prob " + element + " < "+ str(thresh)
      else:
        try:
          probinv = nbmprob.select(name="Time-minimum 2 metre temperature",probabilityTypeName="Probability of event above upper limit",upperLimit=limit)[0]
        except:
          print("Can't find data for this threshold")
          continue
        prob_name="Prob " + element + " > " + str(thresh)
      probdata = probinv.values
    elif element == "temp":
      limit=tval
      if limit < 280:
        try:
          probinv = nbmprob.select(name="2 metre temperature",probabilityTypeName="Probability of event below lower limit",lowerLimit=limit)[0]
        except:
          print("Can't find data for this threshold")
          continue
        prob_name="Prob " + element + " < " + str(thresh)
      else:
        try:
          probinv = nbmprob.select(name="2 metre temperature",probabilityTypeName="Probability of event above upper limit",upperLimit=limit)[0]
        except:
          print("Can't find data for this threshold")
          continue
        prob_name="Prob " + element + " > " + str(thresh)
      probdata = probinv.values
      #prob_name="Prob " + element + " > " + str(thresh)
    elif element == "dewp":
      limit=tval
      if limit < 280:
        try:
          probinv = nbmprob.select(name="2 metre dewpoint temperature",probabilityTypeName="Probability of event below lower limit",lowerLimit=limit)[0]
        except:
          print("Can't find data for this threshold")
          continue
        prob_name="Prob " + element + " < " + str(thresh)
      else:
        try:
          probinv = nbmprob.select(name="2 metre dewpoint temperature",probabilityTypeName="Probability of event above upper limit",upperLimit=limit)[0]
        except:
          print("Can't find data for this threshold")
          continue
      prob_name="Prob " + element + " > " + str(thresh)
      probdata = probinv.values
    elif element == "qpf":
      limit=tval
      try:
        probinv = nbmprob.select(name="Total Precipitation",lengthOfTimeRange=24,probabilityTypeName="Probability of event above upper limit",upperLimit=limit)[0]
      except:
        print("Can't find data for this threshold")
        continue
      probdata = probinv.values
      prob_name="Prob " + element + " > " + str(thresh)
    elif element == "qpf06":
      limit=tval
      try:
        probinv = nbmprob.select(name="Total Precipitation",lengthOfTimeRange=6,probabilityTypeName="Probability of event above upper limit",upperLimit=limit)[0]
      except:
        print("Can't find data for this threshold")
        continue
      probdata = probinv.values
      prob_name=f"Prob {element} > {thresh}"
      print(f"Prob name = {prob_name}")
    elif element == "qpf12":
      limit=tval
      try:
        probinv = nbmprob.select(name="Total Precipitation",lengthOfTimeRange=12,probabilityTypeName="Probability of event above upper limit",upperLimit=limit)[0]
      except:
        print("Can't find data for this threshold")
        continue
      probdata = probinv.values
      prob_name="Prob " + element + " > " + str(thresh)
    elif element == "qpf48":
      limit=tval
      try:
        probinv = nbmprob.select(name="Total Precipitation",lengthOfTimeRange=48,probabilityTypeName="Probability of event above upper limit",upperLimit=limit)[0]
      except:
        print("Can't find data for this threshold")
        continue
      probdata = probinv.values
      prob_name="Prob " + element + " > " + str(thresh)
    elif element == "qpf72":
      limit=tval
      try:
        probinv = nbmprob.select(name="Total Precipitation",lengthOfTimeRange=72,probabilityTypeName="Probability of event above upper limit",upperLimit=limit)[0]
      except:
        print("Can't find data for this threshold")
        continue
      probdata = probinv.values
      prob_name="Prob " + element + " > " + str(thresh)
    elif element == "maxwind":
      limit=tval
      try:
        probinv = nbmprob.select(name="10 metre wind speed",lengthOfTimeRange=24,probabilityTypeName="Probability of event above upper limit",upperLimit=limit)[0]
      except:
        print("Can't find data for this threshold")
        continue
      probdata = probinv.values
      prob_name="Prob " + element + " > " + str(thresh)
    elif element == "snow":
      limit=tval
      try:
        probinv = nbmprob.select(name="unknown",lengthOfTimeRange=24,probabilityTypeName="Probability of event above upper limit",upperLimit=limit)[0]
      except:
        print("Can't find data for this threshold")
        continue
      probdata = probinv.values
      prob_name="Prob " + element + " > " + str(thresh)
    elif element == "vis":
      limit=tval
      try:
        probinv = nbmprob.select(name="Visibility",probabilityTypeName="Probability of event below lower limit",lowerLimit=limit)[0]
      except:
        print("Can't find data for this threshold")
        continue
      probdata = probinv.values
      prob_name="Prob " + element + " < " + str(thresh)
    elif element == "wind":
      limit=tval
      try:
        probinv = nbmprob.select(name="10 metre wind speed",probabilityTypeName="Probability of event above upper limit",upperLimit=limit)[0]
      except:
        print("Can't find data for this threshold")
        continue
      probdata = probinv.values
      prob_name="Prob " + element + " > " + str(thresh)
    elif element == "gust":
      limit=tval
      try:
        probinv = nbmprob.select(name="Instantaneous 10 metre wind gust",probabilityTypeName="Probability of event above upper limit",upperLimit=limit)[0]
      except:
        print("Can't find data for this threshold")
        continue
      probdata = probinv.values
      prob_name="Prob " + element + " > " + str(thresh)
    else:
      raise ValueError (f"Can't extract probabilites for {element} - not configured")
    print("    >>  Extracted " + prob_name)
    probfield[thresh] = probdata

    nbm_coords = filteredObs["NBM_fidx"].values
    prob_values = []
    for i in range(0, len(nbm_coords)):
      prob_value = probdata[nbm_coords[i]]
      prob_values.append(prob_value)
    filteredObs[prob_name] = prob_values
    probfield[thresh] = probdata
nbmprob.close()

valdate=valid_end_datetime
matplotlib.rc('axes',facecolor=background_color,edgecolor=text_color)
fig_valid_date=valdate.strftime('%Y%m%d_%HZ')
valid_title=valdate.strftime('%HZ %a %m-%d-%Y')
nbm_init_title=nbm_init.strftime('%HZ %m-%d-%Y')
nbm_init_string=nbm_init.strftime('%Y%m%d_%H') + "Z"

fileName=f"allprobs_{region}_{element}_{nbm_init_string}_{fig_valid_date}.csv"
filteredObs.to_csv(fileName)

Finally, we draw the maps!  Note that this step takes the longest, as there are several thresholds to plot.  Because of limitations with the free version of Google Colab, we can only generate two plots at a time - most variables have 8-10 thresholds to plot.  When finished, click on the folder in the toolbar to the left to access the files.

In [ ]:
#@title Draw Maps

if cwa_outline:
  # Force re-download and re-extraction of shapefiles if they exist and are problematic
  # The error 'unpack requires a buffer of 2784 bytes' indicates a corrupted or incomplete shapefile.
  # Removing existing files ensures a fresh download.
  for f in glob.glob("shp/w_18mr25.*"):
    if os.path.exists(f):
      os.remove(f)
  if not os.path.exists("shp/w_18mr25.shp"): # Check for the specific shapefile directly
    cwa_url = "https://www.weather.gov/source/gis/Shapefiles/WSOM/w_18mr25.zip" # Use the same URL as before
    if not os.path.exists("shp"):
      os.mkdir("shp")
    urlretrieve(cwa_url, "shp/nws_cwa_outlines.zip") # Retain generic zip name for download if needed
    with zipfile.ZipFile("shp/nws_cwa_outlines.zip", 'r') as zip_ref:
      zip_ref.extractall("shp")

def plotProbsObs(thresh,filteredObs,element,probfield):
  if thresh not in probfield.keys():
    print(f"No data exists for this threshold {thresh}.  Skipping plot...")
    return
  else:
    print(f"Plotting for threshold: {thresh}")
    print("Parent process: " + str(os.getppid()))
    print("Process ID: " + str(os.getpid()))
  south = np.nanmin(filteredObs["lat"]) - 1.0
  east = np.nanmax(filteredObs["lon"]) + 1.0
  north = np.nanmax(filteredObs["lat"]) + 1.0
  width, height = (16,9)
  ratioxy = 16./9.
  width_ratios = [ratioxy]#, 1]
  proj = ccrs.PlateCarree()
  print("Plot dimensions have been set up!")
  plevels=[1,10,20,30,40,50,60,70,80,90,99]

  #add transparent value to bottom of jet colormap
  orig_cmap=plt.cm.get_cmap('jet')
  orig_colors=orig_cmap(np.linspace(0,1,orig_cmap.N))
  landcolor=np.asarray([0.0,0.0,0.0,0.0])
  new_colors=np.vstack((landcolor,orig_colors))
  cmap=ListedColormap(new_colors)

  print("Looking at threshold: " + str(thresh))
  if element == "mint":
    fieldName="Minimum temperature at 2 metres since previous post-processing"
    if thresh <= 32:
      obsplot=filteredObs[filteredObs["ob_mint"] < thresh]
      negPlot=filteredObs[filteredObs["ob_mint"] >= thresh]
      probName="Probability of event below lower limit"
      tdir="< "
    else:
      obsplot=filteredObs[filteredObs["ob_mint"] > thresh]
      negPlot=filteredObs[filteredObs["ob_mint"] <= thresh]
      probName="Probability of event above upper limit"
      tdir="> "
  elif element == "maxt":
    fieldName="Maximum temperature at 2 metres since previous post-processing"
    if thresh < 80:
      obsplot=filteredObs[filteredObs["ob_maxt"] < thresh]
      negPlot=filteredObs[filteredObs["ob_maxt"] >= thresh]
      probName="Probability of event below lower limit"
      tdir="< "
    else:
      obsplot=filteredObs[filteredObs["ob_maxt"] > thresh]
      negPlot=filteredObs[filteredObs["ob_maxt"] <= thresh]
      probName="Probability of event above upper limit"
      tdir="> "
  elif element == "temp":
    fieldName="2 metre temperature"
    if thresh < 80:
      obsplot=filteredObs[filteredObs["ob_temp"] < thresh]
      negPlot=filteredObs[filteredObs["ob_temp"] >= thresh]
      probName="Probability of event below lower limit"
      tdir="< "
    else:
      obsplot=filteredObs[filteredObs["ob_temp"] > thresh]
      negPlot=filteredObs[filteredObs["ob_temp"] <= thresh]
      probName="Probability of event above upper limit"
      tdir="> "
  elif element == "dewp":
    fieldName="2 metre dewpoint temperature"
    if thresh < 40:
      obsplot=filteredObs[filteredObs["ob_dewp"] < thresh]
      negPlot=filteredObs[filteredObs["ob_dewp"] >= thresh]
      probName="Probability of event below lower limit"
      tdir="< "
    else:
      obsplot=filteredObs[filteredObs["ob_dewp"] > thresh]
      negPlot=filteredObs[filteredObs["ob_dewp"] <= thresh]
      probName="Probability of event above upper limit"
      tdir="> "
  elif "qpf" in element:
    fieldName="Total Precipitation"
    obsplot=filteredObs[filteredObs["ob_"+element] > thresh]
    negPlot=filteredObs[filteredObs["ob_"+element] <= thresh]
    probName="Probability of event above upper limit"
    tdir="> "
  elif "snow" in element:
    fieldName="unknown"
    obsplot=filteredObs[filteredObs["ob_snow"] > thresh]
    negPlot=filteredObs[filteredObs["ob_snow"] <= thresh]
    probName="Probability of event above upper limit"
    tdir="> "
  elif element == "maxwind":
    fieldName="10 metre wind speed"
    obsplot=filteredObs[filteredObs["ob_maxwind"] > thresh]
    negPlot=filteredObs[filteredObs["ob_maxwind"] <= thresh]
    probName="Probability of event above upper limit"
    tdir="> "
  elif element == "wind":
    fieldName="10 metre wind speed"
    obsplot=filteredObs[filteredObs["ob_wind"] > thresh]
    negPlot=filteredObs[filteredObs["ob_wind"] <= thresh]
    probName="Probability of event above upper limit"
    tdir="> "
  elif element == "maxgust":
    fieldName="10 metre gust speed"
    obsplot=filteredObs[filteredObs["ob_maxgust"] > thresh]
    negPlot=filteredObs[filteredObs["ob_maxgust"] <= thresh]
    probName="Probability of event above upper limit"
    tdir="> "
  elif element == "gust":
    fieldName="10 metre gust speed"
    obsplot=filteredObs[filteredObs["ob_gust"] > thresh]
    negPlot=filteredObs[filteredObs["ob_gust"] <= thresh]
    probName="Probability of event above upper limit"
    tdir="> "
  elif element == "vis":
    fieldName="Visibility"
    obsplot=filteredObs[filteredObs["ob_vis"] < thresh]
    negPlot=filteredObs[filteredObs["ob_vis"] >= thresh]
    probName="Probability of event below lower limit"
    tdir="< "
  elif element == "fire":
    rhVal=rhThresh[thresh]
    gustVal=gustThresh[thresh]
    windVal=windThresh[thresh]
    fieldName="Fire"
    obsplot=filteredObs[(filteredObs["ob_rh"] < rhVal) & ((filteredObs["ob_wgust_mph"] > gustVal) | (filteredObs["ob_wspd_mph"] > windVal))]
    negPlot=filteredObs[(filteredObs["ob_rh"] >= rhVal) & ((filteredObs["ob_wgust_mph"] <= gustVal) | (filteredObs["ob_wspd_mph"] <= windVal))]
    probName="Join Fire Wx Probability"
    #tdir="> "
  else:
    raise ValueError (f"Can't extract probabilites for {element} - not configured")
  print("    >>  Extracted " + probName)
  #if tdir == "> ":
  #  probfield = nbmprob.select(name=fieldName,probabilityTypeName=probName,upperLimit=thresh)[0]
  #  probdata = probfield.values
  #else:
  #  probfield = nbmprob.select(name=fieldName,probabilityTypeName=probName,lowerLimit=thresh)[0]
  #  probdata = probfield.values
  fig = plt.figure(constrained_layout=True, figsize=(width,height), facecolor=background_color, frameon=True, dpi=150)
  grid = fig.add_gridspec(1,1, hspace=0.1, width_ratios=width_ratios, height_ratios = [1], wspace=0.2)
  ax1 = fig.add_subplot(grid[0,0], projection=ccrs.Mercator())
  ax1.set_anchor('N')
  ax1.set_facecolor(background_color)
  ax1.set_extent([np.nanmin(filteredObs["lon"]) - 1.0, east, south, north], crs=proj)
  ax1.add_feature(cfeature.OCEAN, edgecolor='none', facecolor=map_water_color, zorder=-2)
  ax1.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='none', facecolor=map_land_color, zorder=-1))
  #ax1.add_feature(cfeature.LAKES, edgecolor='none', facecolor='#272727', zorder=0)
  ax1.add_feature(cfeature.NaturalEarthFeature('physical', 'lakes', '10m', edgecolor='none', facecolor=map_water_color, zorder=0))
  ax1.add_feature(cfeature.BORDERS, edgecolor=map_border_color, facecolor='none', linewidth=2, zorder=1)
  #ax1.add_feature(cfeature.NaturalEarthFeature('cultural', 'countries', '50m', edgecolor=map_border_color, facecolor='none', linewidth=2, zorder=2))
  ax1.add_feature(cfeature.NaturalEarthFeature('cultural', 'admin_1_states_provinces_lines', '50m', edgecolor=map_border_color, facecolor='none', linewidth=1, zorder=2))
  if cwa_outline:
    try:
      cwa_shapes=gpd.read_file("shp/w_18mr25.shp") # Changed to w_18mr25.shp
      cwaList=cwa_id.split(',')
      cwa_subset=cwa_shapes[cwa_shapes['WFO'].isin(cwaList)]
      print("Drawing CWA outlines...")
      cwa_feature=ShapelyFeature(cwa_subset.geometry,ccrs.PlateCarree(), edgecolor='black', facecolor='none', linewidth=1, linestyle='-', zorder=4)
      ax1.add_feature(cwa_feature, zorder=4)
    except:
      print("Aw shucks, no CWA boundaries for you. Sorry bout that.")
  print("Drawing obs...")
  hits=ax1.scatter(obsplot["lon"].values,obsplot["lat"].values,color="white",s=4,transform=proj,label="Observed Event",zorder=5)
  misses=ax1.scatter(negPlot["lon"].values,negPlot["lat"].values,color="black",s=4,transform=proj,label="Observed Non-Event",zorder=5)
  print("Drawing Probs...")
  probs=ax1.pcolormesh(nbmlons,nbmlats,probfield[thresh],vmin=1.,vmax=99.,cmap=cmap,transform=proj,rasterized=True,snap=True,zorder=1)
  cbar=fig.colorbar(probs,boundaries=plevels,ticks=plevels)
  cbar.set_label("% Probability",color=text_color)
  cbar.ax.tick_params(color=text_color,which='both',labelcolor=text_color)
  print("Drawing titles")
  if element == "fire":
    if math.isnan(gustVal):
      fig.text(0.5,1.05,f"Obs and Probs: NBM Operational Joit Fire Wx Probs: RH < {rhVal:.0f} and Wind Speed > {windVal:.0f}",horizontalalignment='center', verticalalignment='bottom', weight='bold',fontsize=20,color=text_color)
    elif math.isnan(windVal):
      fig.text(0.5,1.05,f"Obs and Probs: NBM Operational Joit Fire Wx Probs: RH < {rhVal:.0f} and Gust Speed > {gustVal:.0f}",horizontalalignment='center', verticalalignment='bottom', weight='bold',fontsize=20,color=text_color)
    else:
      raise ValueError (f"For Fire Wx Prob Plotting: Both gustVal {gustVal} and windVal {windVal} are listed.  One should be NaN.")
  else:
    fig.text(0.5,1.05,"Obs and Probs: NBM Operational " + element + " " + tdir + str(thresh),horizontalalignment='center', verticalalignment='bottom', weight='bold',fontsize=20,color=text_color)
  fig.text(0.5,1.05,"Valid ending at: " + valid_title + " | NBM Init: " + nbm_init_title,horizontalalignment='center',verticalalignment='top', fontsize=16,color=text_color)
  if element == "fire":
    if math.isnan(gustVal):
      figname=f"obs_and_probs_{region}_fire_RH{rhVal:.0f}_wind{windVal:.0f}_{nbm_init_string}_{fig_valid_date}.png"
    else:
      figname=f"obs_and_probs_{region}_fire_RH{rhVal:.0f}_gust{gustVal:.0f}_{nbm_init_string}_{fig_valid_date}.png"
  else:
    figname=f"obs_and_probs_{region}_{element}_{thresh}_{nbm_init_string}_{fig_valid_date}.png"
  print("Saving figure as " + figname)
  plt.savefig(figname,bbox_inches='tight',pad_inches=0.2,dpi='figure')
  print("   >Done! Saved plot as " + figname)
  plt.close()
  return

manager=Manager()
args=[]
if element == "fire":
  procsToUse=len(rhThresh)
  with Pool(processes=min(procsToUse,os.cpu_count())) as pool:
    pool.starmap(plotProbsObs,[(thresh,filteredObs,element,probfield) for thresh in rhThresh.keys()])
else:
  procsToUse=len(idthresh)
  with Pool(processes=min(procsToUse,os.cpu_count())) as pool:
    pool.starmap(plotProbsObs,[(thresh,filteredObs,element,probfield) for thresh in idthresh.keys()])
